In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv1D,
    MaxPooling1D,
    GRU,
    Dropout,
    Dense
)

from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

In [ ]:
import numpy as np

X_train = np.load("C:\\Users\\ASUS\\Desktop\\Narmada Project\\Data\\Processed\\sequences\\X_train.npy")
y_train = np.load("C:\\Users\\ASUS\\Desktop\\Narmada Project\\Data\\Processed\\sequences\\y_train.npy")

X_val = np.load("C:\\Users\\ASUS\\Desktop\\Narmada Project\\Data\\Processed\\sequences\\X_val.npy")
y_val = np.load("C:\\Users\\ASUS\\Desktop\\Narmada Project\\Data\\Processed\\sequences\\y_val.npy")

X_test = np.load("C:\\Users\\ASUS\\Desktop\\Narmada Project\\Data\\Processed\\sequences\\X_test.npy")
y_test = np.load("C:\\Users\\ASUS\\Desktop\\Narmada Project\\Data\\Processed\\sequences\\y_test.npy")

In [ ]:
y_train_flow = y_train[:, 1]
y_val_flow   = y_val[:, 1]
y_test_flow  = y_test[:, 1]

print(X_train.shape)
print(y_train_flow.shape)

In [ ]:
model = Sequential([

    # CNN Layer
    Conv1D(
        filters=64,
        kernel_size=3,
        activation='relu',
        input_shape=(30, 16)
    ),

    # Pooling Layer
    MaxPooling1D(pool_size=2),

    # GRU Layer
    GRU(64),

    # Regularization
    Dropout(0.2),

    # Dense Hidden Layer
    Dense(32, activation='relu'),

    # Output Layer
    Dense(1)

])


In [ ]:
model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

In [ ]:
model.summary()

In [ ]:
early_stop = EarlyStopping(

    monitor='val_loss',

    patience=8,

    restore_best_weights=True

)

In [ ]:
history = model.fit(

    X_train,
    y_train_flow,

    validation_data=(X_val, y_val_flow),

    epochs=50,

    batch_size=32,

    callbacks=[early_stop],

    verbose=1
)

In [ ]:
y_pred = model.predict(X_test)

# Flatten predictions
y_pred = y_pred.flatten()

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test_flow, y_pred))

mae = mean_absolute_error(y_test_flow, y_pred)

r2 = r2_score(y_test_flow, y_pred)

print("\nCNN-GRU FLOW METRICS")
print("RMSE:", rmse)
print("MAE :", mae)
print("R2  :", r2)

In [ ]:

print("\nFlow Variance:", np.var(y_test_flow))

In [ ]:
plt.figure(figsize=(14,6))

plt.plot(
    y_test_flow[:500],
    label='Actual Flow'
)

plt.plot(
    y_pred[:500],
    label='Predicted Flow'
)

plt.title("CNN-GRU Flow Prediction")

plt.xlabel("Time")

plt.ylabel("Flow")

plt.legend()

plt.show()

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(
    history.history['loss'],
    label='Train Loss'
)

plt.plot(
    history.history['val_loss'],
    label='Validation Loss'
)

plt.title("CNN-GRU Flow Loss")

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.legend()

plt.show()

In [ ]:
model.save("cnn_gru_flow_model.keras")

print("\nModel Saved Successfully!")